In [3]:
# Setup Environment Variables and LangSmith Tracing
import os
from dotenv import load_dotenv

# Load keys from .env file
load_dotenv()

# LangSmith Tracing Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "ai-security-observability-lab"

print("✅ LangSmith Tracing configured for project:", os.environ.get("LANGCHAIN_PROJECT"))
print("OpenAI API Key set:", bool(os.environ.get("OPENAI_API_KEY")))
print("LangSmith API Key set:", bool(os.environ.get("LANGSMITH_API_KEY")))

✅ LangSmith Tracing configured for project: ai-security-observability-lab
OpenAI API Key set: True
LangSmith API Key set: True


In [4]:
import time
from langsmith import traceable
@traceable(run_type="retriever")
def search_docs(question):
    time.sleep(0.2)
    return ["refund_policy.md","faq.md"]

@traceable(run_type="llm")
def LLM_Fake(question,docs):
    time.sleep(0.4)
    return "Based on " + str(len(docs))+ "documents, here is the answer."
@traceable
def pipeline(question):
    docs=search_docs(question)
    answer=LLM_Fake(question,docs)
    return answer
print(pipeline("what is refund police?"))




Based on 2documents, here is the answer.


Building a Rag app

In [5]:
from langchain_openai import ChatOpenAI
from openai.types import model
llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)
reply=llm.invoke("reply with exactly: langsmith is listening")
print(reply.content)

langsmith is listening


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader=PyPDFLoader("enterprise_ai_security_policy.pdf")
Original_docs=loader.load()
print(Original_docs)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T19:44:46-04:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T19:44:46-04:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'enterprise_ai_security_policy.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='■■ Enterprise AI Security & Incident Response\nStandard\nDocument Version: 3.2.0 | Classification: CONFIDENTIAL - INTERNAL USE ONLY | Effective Date: 2026-Q1\n1. Purpose, Scope & Governance\nThis document establishes the mandatory security controls, operational guardrails, and compliance standards\ngoverning all Generative AI applications, Large Language Model (LLM) agents, and Retrieval-Augmented Generation\n(RAG) pipelines deployed within the enterprise infrastructure.\n2. Authentication, API Key Lifecycle & Access Control (RBAC)\n \x7f Rule AC-101 (MFA Mandate): Multi-Factor Authen

In [8]:
text_Splitter=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=80)
docs=text_Splitter.split_documents(Original_docs)
len(docs)
print(docs[0].page_content)
print(docs[0].metadata)

■■ Enterprise AI Security & Incident Response
Standard
Document Version: 3.2.0 | Classification: CONFIDENTIAL - INTERNAL USE ONLY | Effective Date: 2026-Q1
1. Purpose, Scope & Governance
This document establishes the mandatory security controls, operational guardrails, and compliance standards
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T19:44:46-04:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T19:44:46-04:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'enterprise_ai_security_policy.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [9]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=InMemoryVectorStore.from_documents(docs,embedding=embeddings)

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke("What is the SLA for isolating a breached model endpoint?")
for i, doc in enumerate(retrieved_docs):
    print(f"--- CHUNK {i+1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

--- CHUNK 1 ---
5. High-Severity Incident Response & Emergency Containment (IRP)
  Protocol IR-401 (Isolation SLA): In the event of a confirmed model exploit, data exfiltration breach, or autonomous
agent loop runaway, the SecOps team must isolate the model endpoint within 5 minutes.
 Protocol IR-402 (SOC Escalation): Security Operations Center (SOC) must be automatically paged via PagerDuty
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T19:44:46-04:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T19:44:46-04:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'enterprise_ai_security_policy.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
--- CHUNK 2 ---
instructions.
 Rule PI-203 (Canary Tokens): System prompts must embed unique cryptographic canary tokens. If a model output
contains the canary token, the session must be immediately terminated and fla

In [11]:
# 4. Prompt Template & LCEL Chain Assembly
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Helper function to format retrieved documents
def format_docs(retrieved_documents):
    return "\n\n".join(doc.page_content for doc in retrieved_documents)

# Security prompt template with strict citation rules
prompt_template = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert Enterprise AI Security Analyst.\n"
        "Answer the question using ONLY the provided security context below.\n"
        "If the information is not in the context, say: 'I cannot find this in the policy.'\n"
        "Always cite the exact rule code (e.g., Rule AC-102, Protocol IR-401, Rule OB-502)."
    )),
    ("human", "Security Context:\n{context}\n\nQuestion: {question}")
])

# LLM setup (temperature=0 for strict deterministic compliance answers)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Assemble the complete LCEL RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("🚀 Complete Enterprise Security RAG Chain is ready with automatic LangSmith tracing!")


🚀 Complete Enterprise Security RAG Chain is ready with automatic LangSmith tracing!


In [12]:
# 9. Two Small Practical Functions with @traceable (Self-Contained)
import os
from dotenv import load_dotenv
from langsmith import traceable
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

load_dotenv(override=True)


if 'retriever' not in globals():
    loader = PyPDFLoader("enterprise_ai_security_policy.pdf")
    docs = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80).split_documents(loader.load())
    vector_store = InMemoryVectorStore.from_documents(docs, embedding=OpenAIEmbeddings(model="text-embedding-3-small"))
    retriever = vector_store.as_retriever(search_kwargs={"k": 2})

@traceable(run_type="tool", name="mask_api_key")
def mask_api_key(api_key: str) -> str:
    """Masks sensitive API key to prevent secret leakage."""
    if len(api_key) > 8:
        return api_key[:4] + "..." + api_key[-4:]
    return "[REDACTED]"

@traceable(run_type="chain", name="quick_policy_check")
def quick_policy_check(question: str, user_api_key: str) -> dict:
    """Performs a fast security check using our tool and real PDF retriever."""
    safe_key = mask_api_key(user_api_key)
    matching_docs = retriever.invoke(question)
    
    return {
        "masked_key": safe_key,
        "question": question,
        "matched_policy_rule": matching_docs[0].page_content[:120] + "..."
    }


result = quick_policy_check(
    question="What is the key rotation policy?", 
    user_api_key="sk-proj-9876543210abcdef"
)

print("=" * 60)
print("✅ Result from quick_policy_check:")
print("Masked Key:", result["masked_key"])
print("Question:", result["question"])
print("Matched Policy:", result["matched_policy_rule"])
print("=" * 60)
print("👉 Open LangSmith to see the 2-span trace: quick_policy_check -> mask_api_key!")


✅ Result from quick_policy_check:
Masked Key: sk-p...cdef
Question: What is the key rotation policy?
Matched Policy: endpoints, vector databases, and LangSmith observability dashboards.
 Rule AC-102 (Key Rotation): All AI API keys (Open...
👉 Open LangSmith to see the 2-span trace: quick_policy_check -> mask_api_key!
